In [1]:
###################################
### xml converter for HuWN and ####
### potentially other xml files ###
###################################

# we have:
# Hungarian HuWN

In [2]:
###!!!! TO DO !!!!###
### CLEAN THIS UP ###
###!!!!!!!!!!!!!!!###

from xml.sax.handler import feature_external_ges

#########################
### imports and stuff ###
#########################

### pip install bs4
### pip install lxml

### for xml files: ElementTree? ###
from pathlib import Path

import requests
import lxml
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, element

In [3]:
### this one is just for visualization and can be substituted with relevant function later ###
### used in

def show_xml(element, level=0, count=100):
    c = 0
    indent = "  " * level
    text = (element.text or "").strip()
    attrs = element.attrib

    if attrs:
        attr_str = ", ".join([f"{k}={v}" for k, v in attrs.items()])
        print(f"{indent}- {element.tag} [{attr_str}]")
    else:
        print(f"{indent}- {element.tag}: {text}")

    for child in element:
        show_xml(child, level + 1)

        if c >= count:
            break

        c += 1

In [4]:
### so this function just creates a dict with ID: literal
### - for the structure of the HuWN xml file
### NL is a label to mark non-lexicality, these entries can be excluded

def xml_id_dict(path, lit=None):

    if lit is None:
        lit = {}

    tree = ET.parse(path)
    root = tree.getroot()

    for elem in root.findall('SYNSET'):
        if elem.find('NL') is None:
            lit[elem.find('ID').text] = []
            for l in elem.find('SYNONYM').findall('LITERAL'):
                lit[elem.find('ID').text].append(l.text)

    return lit

In [5]:
### same procedure as before, just with literal: ID, ALL the literals

def xml_lit_dict(path, lit=None):

    if lit is None:
        lit = {}

    tree = ET.parse(path)
    root = tree.getroot()

    for elem in root.findall('SYNSET'):
        if elem.find('NL') is None:
            for l in elem.find('SYNONYM').findall('LITERAL'):
                if l not in lit:
                    lit[l.text] = []
                    lit[l.text].append(elem.find('ID').text)
                else: lit[l.text].append(elem.find('ID').text)
        else:
            print(f'{elem.find("ID").text}: {elem.find("SYNONYM").find("LITERAL").text}')
    return lit

In [6]:
### for PC, use this one:
#huWnDict = xml_lit_dict(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml')

### for laptop, this one:
#huWnDict = xml_lit_dict(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/HuWN_final4.xml')

huWnDict2 = xml_id_dict(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml')

#some tries to figure out ID.....
for key in huWnDict2:
    if key.__contains__("2452-"):
        print(huWnDict2.get(key).__str__().replace('[', '').replace(']', '').replace("'", '').replace("'", ''))

#print(hunDict2.get('HuWN-747815818-n'))
#print(hunDict2)

vérengzés, vérfürdő, vérontás, öldöklés
biológiai csoport
mérsékelt égöv
tányér
Tomázia
Upor


In [7]:
###  function that takes in an ID/literal and returns
### relevant info (literal, hypernyms, hyponyms, synonyms, pos)


def xml_getSynset(path, lit='', id=''):
    if lit == '' and id == '':
        return 'provide at least either ID or LITERAL'

    tree = ET.parse(path)
    root = tree.getroot()
    id_dct = {}

    if id == '':
        for elem in root.findall('SYNSET'):
            lits = elem.find('SYNONYM').findall('LITERAL')
            for l in lits:
                if lit == l.text:
                    if elem.find('ILR').find('TYPE').text == 'hypernym':
                        id_dct[elem.find('ID').text] = [elem.find('ILR').text]
                    else:
                        id_dct[elem.find('ID').text] = ['']
                    id_dct[elem.find('ID').text].append(l.text)
                    # we need to include synonyms here with _for syn in lits: [...].append(syn.text)
                    synlist = []
                    for syn in lits:
                        if syn.text != lit:
                            synlist += [syn.text]
                    id_dct[elem.find('ID').text].append(synlist)
                    # just to separate this one from the others for now
                    id_dct[elem.find('ID').text].append(elem.find('POS').text)
                    if elem.find('ILR').find('TYPE').text == 'hyponym':
                        id_dct[elem.find('ID').text].append(elem.find('ILR').text)

    elif lit == '':
        for elem in root.findall('SYNSET'):
            lits = elem.find('SYNONYM').findall('LITERAL')
            if id == elem.find('ID'):
            #if elem.find('ID').__contains__(id):
                if elem.find('ILR').find('TYPE').text == 'hypernym':
                        id_dct[id] = [elem.find('ILR').text]
                else:
                    id_dct[id] = ['']
                for l in lits:
                    id_dct[id].append(l.text)
                id_dct[id].append(elem.find('POS').text)
                if elem.find('ILR').find('TYPE').text == 'hyponym':
                    id_dct[id].append(elem.find('ILR').text)

    return id_dct


In [8]:
### for PC, use this one:
#xml_getSynset(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml', lit='hida')

### for laptop, this one:
xml_getSynset(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml', lit='növény')
#xml_getSynset(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/HuWN_final4.xml', lit='plant')

{'ENG20-00014510-n': ['ENG20-00003226-n', 'növény', [], 'n']}